# scPRINT2 GRN Inference — PI-Revised Genelist Strategy

## What changed from the previous notebook (`scprint_grn_clean.ipynb`)

The previous notebook built one **shared genelist** across all four states:
```
genelist = union(DE markers across all 4 states) ∪ all expressed TFs
```
This ensured the same gene axis across runs, but it mixed state-specific biology together.

**The PI's revised strategy is state-specific genelists:**
```
For CD4_rest → genelist = pySCENIC resting TFs for CD4  (deltaRSS < 0)
                         ∪ significant DEGs comparing CD4_rest vs CD4_act (padj < 0.05)

For CD4_act  → genelist = pySCENIC activation TFs for CD4 (deltaRSS > 0)
                         ∪ significant DEGs comparing CD4_rest vs CD4_act (padj < 0.05)

For CD8_rest → same logic but using CD8 deltaRSS and CD8 rest vs act DEGs
For CD8_act  → same logic but using CD8 deltaRSS and CD8 rest vs act DEGs
```

**Why this is biologically better:**
- The resting genelist focuses on TFs with higher regulon specificity in resting T cells,
  paired with genes that actually change when T cells transition to quiescence.
  This captures the *quiescence program* specifically.
- The activation genelist focuses on TFs whose programs are more specific to activated T cells,
  paired with the DEGs that mark that activation response.
  This captures the *activation program* specifically.
- Using `how='some'` tells scPRINT2 to build its attention network **over exactly these genes**,
  so the resulting edge weights are concentrated on biologically-relevant TF→target pairs.

## ⚠️ Matrix directionality — READ BEFORE RUNNING

scPRINT2's attention mechanism is based on the transformer's query-key product.
In a transformer, the attention matrix entry [i, j] = how much gene i (as **query**)
attends to gene j (as **key**). In the GRN interpretation:

> **GRN[i, j] = the weight with which gene i is regulated by gene j**
> In other words: **column j regulates row i** (key → query, or source → target)

This means in `grn.varp['GRN']`:
- **Rows are targets** (the gene being regulated)
- **Columns are regulators/TFs** (the gene doing the regulating)

So the correct extraction is:
```python
rows, cols = np.where((dense > threshold) & np.outer(~is_tf, is_tf))
#                                                     ↑ target   ↑ TF (column = regulator)
edges['TF']     = symbols[cols]   # column = regulator
edges['target'] = symbols[rows]   # row    = target
```

**The previous notebook used `np.outer(is_tf, ~is_tf)` which assumed rows=TF, cols=target
— this was inverted.** This notebook corrects that, and includes a symmetry check so you
can verify the directionality is consistent with known biology (e.g., FOSB should have
outgoing edges to AP-1 target genes in activated T cells, not incoming ones).

Note: if the scPRINT2 source code changes this convention, re-read the `GNInfer.__call__`
method and look for how `attn_weights` are assigned — the axis labeled `query` is the
regulated gene (row), the axis labeled `key` is the regulator (column).

In [ ]:
# ── STEP 2 of 2: Install remaining packages + imports (run AFTER restart) ─────
#
# numpy<2.0 is now locked in. Install the rest of the stack in one shot to
# avoid subsequent uninstalls/reinstalls fighting each other.
# The trailing `numpy<2.0 --upgrade` re-asserts the pin in case any dep
# silently pulled numpy 2.x back in.

!pip install -q scprint2 lamindb bionty scdataloader scanpy anndata scipy huggingface_hub grnndata bengrn
!pip install -q "numpy<2.0" --upgrade

# Verify pin held
import numpy as np
print(f'numpy version: {np.__version__}')  # must be 1.x

# ── Imports ───────────────────────────────────────────────────────────────────
import os, gc, importlib
import torch
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import bionty as bt

from huggingface_hub import hf_hub_download
from scdataloader.utils import populate_my_ontology, load_genes
from grnndata import utils as grnutils

torch.set_float32_matmul_precision('medium')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# ── CELL 3: Mount Drive + lamin auth ──────────────────────────────────────────
# lamindb is scPRINT2's gene ontology backend. It needs a one-time init per
# runtime. Use your EMAIL on a completely fresh compute environment; the handle
# (preelshh) works after first login on a given machine.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
shutil.rmtree('/content/lamindb-storage', ignore_errors=True)
shutil.rmtree('/root/.lamin', ignore_errors=True)
os.makedirs('/root/.lamin', exist_ok=True)

!lamin login preelshh   # prompts for API key from lamin.ai → Settings → API keys

import lamindb as ln
ln.setup.init(storage='/content/lamindb-storage', name='scprintGRN_pi_revised', modules='bionty')
print('lamin ready ✓')

Mounted at /content/drive
✗ Use your email for your first login in a compute environment. After that, you can use your handle.
! using anonymous user (to identify, call: lamin login)
→ initialized lamindb: anonymous/scprintGRN_pi_revised
lamin ready ✓


/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:218: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  isinstance(v, types.ModuleType)
/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:225: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  elif hasattr(v, "__module__") and getattr(v, "__module__", None):
/usr/local/lib/python3.12/dist-packages/lamindb_setup/_connect_instance.py:226: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  class_module = v.__module__


In [ ]:
# ── CELL 4: Imports ───────────────────────────────────────────────────────────
import os, gc, importlib, gzip, requests
import torch
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import bionty as bt

from huggingface_hub import hf_hub_download
from scdataloader.utils import populate_my_ontology, load_genes
from grnndata import utils as grnutils

torch.set_float32_matmul_precision('medium')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

torch: 2.8.0+cu128 | CUDA: True


In [ ]:
# ── CELL 5: Config ────────────────────────────────────────────────────────────
# All file paths and hyperparameters in one place. Edit here only.

CKPT_PATH       = '/content/small-v2.ckpt'
DROPBOX_URL     = ('https://www.dropbox.com/scl/fi/e38rl88uk3m5ux4v29vdt/'
                   'compressed_cleaned_labelled_t_cell_data.gz'
                   '?rlkey=e0ww30defqgh8dhyak28ig8lr&e=1&dl=1')
COMPRESSED_PATH = '/content/tcell_cache/tcell_raw.gz'
H5AD_PATH       = '/content/tcell_cache/tcell_raw.h5ad'

# Lambert et al. 2018 human TF list (1,639 TFs)
TF_CSV_PATH     = '/content/drive/MyDrive/benchmarking_paper/datasets/humanTFs_1639_clean.csv'

# pySCENIC deltaRSS table: 4 rows (CD4_rest, CD4_act, CD8_rest, CD8_act) × 235 TF columns
# Values are RSS_act - RSS_rest per TF per lineage (computed in the pySCENIC notebook).
SCENIC_RSS_PATH = '/content/drive/MyDrive/benchmarking_paper/datasets/pyscenic/pruned_rss_score_matrix_final.csv'

# Pre-computed scanpy DE markers per lineage (CD4: rest vs act, CD8: rest vs act)
# Format: columns [state, names, pvals_adj, logfoldchanges]
# Generated by running sc.tl.rank_genes_groups on CD4 cells (rest vs act)
# and separately on CD8 cells (rest vs act). Must be uploaded to Colab before running.
DE_CD4_PATH     = '/content/de_cd4_rest_vs_act.csv'   # CD4 rest-vs-act DE results
DE_CD8_PATH     = '/content/de_cd8_rest_vs_act.csv'   # CD8 rest-vs-act DE results

# Output directory on Drive (created if absent)
GRN_OUT_DIR     = ('/content/drive/MyDrive/benchmarking_paper/datasets/'
                   'scPRINT/grn_outputs_pi_revisedv3')

STATES          = ['CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act']
MAX_CELLS       = 300    # max cells subsampled per state for GPU inference
BATCH_SIZE      = 16
CELLS_PER_STATE = 2000   # cells per state for neighbor graph computation
WEIGHT_THRESHOLD = 0.01  # min attention weight to keep an edge

os.makedirs('/content/tcell_cache', exist_ok=True)
os.makedirs(GRN_OUT_DIR, exist_ok=True)
print('Config set ✓')
print(f'Output dir: {GRN_OUT_DIR}')

Config set ✓
Output dir: /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3


In [ ]:
# ── CELL 6: Download + extract raw T cell data from Dropbox ───────────────────
# We use the raw (unscaled) h5ad from Dropbox, NOT the z-scored HVG-filtered
# file on Drive. scPRINT2 normalizes internally, so it needs raw counts.
# Cached files are reused across sessions; delete manually to re-download.

if not os.path.exists(COMPRESSED_PATH):
    print('Downloading from Dropbox...')
    with requests.get(DROPBOX_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total_mb = int(r.headers.get('content-length', 0)) / 1024**2
        downloaded = 0
        with open(COMPRESSED_PATH, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    print(f'  {downloaded/1024**2:.0f} / {total_mb:.0f} MB', end='\r')
    print(f'\nDownloaded ✓')
else:
    print(f'Cached .gz found ✓  ({os.path.getsize(COMPRESSED_PATH)/1024**2:.0f} MB)')

if not os.path.exists(H5AD_PATH):
    print('Extracting .gz → .h5ad ...')
    with gzip.open(COMPRESSED_PATH, 'rb') as f_in, open(H5AD_PATH, 'wb') as f_out:
        extracted = 0
        while True:
            chunk = f_in.read(10 * 1024**2)
            if not chunk: break
            f_out.write(chunk)
            extracted += len(chunk)
            print(f'  Extracted: {extracted/1024**2:.0f} MB', end='\r')
    print(f'\nExtracted ✓  ({os.path.getsize(H5AD_PATH)/1024**3:.2f} GB)')
else:
    print(f'Cached .h5ad found ✓  ({os.path.getsize(H5AD_PATH)/1024**3:.2f} GB)')

NameError: name 'os' is not defined

In [ ]:
# ── CELL 7: Load data ─────────────────────────────────────────────────────────
# This dataset has 47,726 cells × 58,828 genes (raw counts).
# We build the group4 column (CD_Status + Stimulation) for cell state splitting.
# NOTE: obs columns may be lowercase ('cd_status', 'stimulation') depending on
# which version of the file you have. Check and adjust below if needed.

adata = sc.read_h5ad(H5AD_PATH)
print(f'Loaded: {adata.shape}')
print(f'obs columns: {list(adata.obs.columns[:10])} ...')

# Build four-way state label
# Adjust column names if your file uses different capitalization
try:
    adata.obs['group4'] = (
        adata.obs['cd_status'].astype(str) + '_' +
        adata.obs['stimulation'].astype(str)
    )
except KeyError:
    # Some versions use CD_Status / Stimulation
    adata.obs['group4'] = (
        adata.obs['CD_Status'].astype(str) + '_' +
        adata.obs['Stimulation'].astype(str)
    )

print('Group counts:')
print(adata.obs['group4'].value_counts())

# Required scPRINT2 metadata columns — add defaults for any that are missing
for col, val in {
    'organism_ontology_term_id':                'NCBITaxon:9606',
    'assay_ontology_term_id':                   'EFO:0009922',
    'self_reported_ethnicity_ontology_term_id': 'unknown',
    'disease_ontology_term_id':                 'PATO:0000461',
    'sex_ontology_term_id':                     'unknown',
}.items():
    if col not in adata.obs.columns:
        adata.obs[col] = val
print('Metadata columns set ✓')

Loaded: (47726, 58828)
obs columns: ['stimulation', 'cd_status'] ...
Group counts:
group4
CD4_act     18144
CD4_rest    16808
CD8_rest     6669
CD8_act      6105
Name: count, dtype: int64
Metadata columns set ✓


In [ ]:
# ── CELL 8: Load TF reference list (Lambert et al. 2018) ──────────────────────
# This is our canonical set of 1,639 human transcription factors.
# We use it to (a) identify which genes in the dataset are TFs, and
# (b) build state-specific genelists from pySCENIC candidates.

tf_df = pd.read_csv(TF_CSV_PATH)
all_tf_symbols = set(tf_df['gene_symbol'])
expressed_tfs  = all_tf_symbols & set(adata.var_names)

print(f'Total human TFs (Lambert 2018):  {len(all_tf_symbols)}')
print(f'TFs present in this dataset:     {len(expressed_tfs)}')
print(f'TFs NOT in this dataset:         {len(all_tf_symbols) - len(expressed_tfs)}')

Total human TFs (Lambert 2018):  1639
TFs present in this dataset:     1627
TFs NOT in this dataset:         12


## Building State-Specific Genelists

The PI's instruction is to use **pySCENIC candidate TFs** as the TF component
of each state's genelist, rather than all expressed TFs. This is better because:

- For **resting** states, we focus on TFs whose regulon specificity score is
  **higher in rest than act** (deltaRSS < 0), i.e., TFs that maintain the quiescent program.
- For **activation** states, we focus on TFs with **higher specificity in act** (deltaRSS > 0),
  i.e., TFs that drive the activation response.

This gives scPRINT2 a biologically-informed gene universe for each run,
rather than a generic set of expressed genes.

In [ ]:
# ── CELL 9: Load pySCENIC RSS matrix and extract state-specific TF candidates ──
# The RSS matrix (pruned_rss_score_matrix_final.csv) has:
#   rows    = cell states (CD4_rest, CD4_act, CD8_rest, CD8_act)
#   columns = regulon names like 'FOSB(+)', 'KLF2(+)', ...
#   values  = RSS score (how specific is this TF's program to that cell state)
#
# We compute deltaRSS = RSS_act - RSS_rest for each lineage (CD4, CD8).
# Positive deltaRSS → TF is more specific to activated T cells (activation TF)
# Negative deltaRSS → TF is more specific to resting T cells (quiescence TF)

rss = pd.read_csv(SCENIC_RSS_PATH, index_col=0)
print('RSS matrix shape:', rss.shape)
print('Row labels:', rss.index.tolist())
print('First 5 regulon columns:', rss.columns[:5].tolist())

# Strip the (+) or (-) suffix from regulon names to get plain TF symbols
# e.g. 'FOSB(+)' → 'FOSB'
regulon_to_tf = {col: col.split('(')[0] for col in rss.columns}
rss_clean = rss.rename(columns=regulon_to_tf)

# Compute deltaRSS per lineage
# CD4: act − rest
delta_cd4 = rss_clean.loc['CD4_act'] - rss_clean.loc['CD4_rest']
# CD8: act − rest
delta_cd8 = rss_clean.loc['CD8_act'] - rss_clean.loc['CD8_rest']

print(f'\nCD4 deltaRSS range: [{delta_cd4.min():.4f}, {delta_cd4.max():.4f}]')
print(f'CD8 deltaRSS range: [{delta_cd8.min():.4f}, {delta_cd8.max():.4f}]')

# Extract TF candidates per state:
#   Resting TFs  = deltaRSS < 0 (regulon more specific to rest)
#   Activation TFs = deltaRSS > 0 (regulon more specific to act)
# Only keep TFs that are actually in our dataset
pyscenic_tfs = {
    'CD4_rest': set(delta_cd4[delta_cd4 < 0].index) & expressed_tfs,
    'CD4_act':  set(delta_cd4[delta_cd4 > 0].index) & expressed_tfs,
    'CD8_rest': set(delta_cd8[delta_cd8 < 0].index) & expressed_tfs,
    'CD8_act':  set(delta_cd8[delta_cd8 > 0].index) & expressed_tfs,
}

print('\n=== pySCENIC TF candidates per state (expressed in dataset) ===')
for state, tfs in pyscenic_tfs.items():
    print(f'{state:<12}: {len(tfs):3d} TFs')
    print(f'  e.g. {sorted(tfs)[:8]}')

RSS matrix shape: (4, 235)
Row labels: ['CD4_act', 'CD8_rest', 'CD8_act', 'CD4_rest']
First 5 regulon columns: ['ARID3A(+)', 'ARNTL(+)', 'ATF1(+)', 'ATF2(+)', 'ATF3(+)']

CD4 deltaRSS range: [-0.2041, 0.1094]
CD8 deltaRSS range: [-0.0802, 0.0387]

=== pySCENIC TF candidates per state (expressed in dataset) ===
CD4_rest    : 133 TFs
  e.g. ['ARID3A', 'ATF2', 'ATF6', 'ATF6B', 'BACH1', 'BACH2', 'BCL11A', 'BPTF']
CD4_act     :  75 TFs
  e.g. ['ARNTL', 'ATF1', 'ATF3', 'ATF4', 'ATF5', 'BATF', 'BATF3', 'CEBPB']
CD8_rest    : 181 TFs
  e.g. ['ARID3A', 'ARNTL', 'ATF1', 'ATF2', 'ATF3', 'ATF5', 'ATF6', 'ATF6B']
CD8_act     :  27 TFs
  e.g. ['ATF4', 'BACH1', 'BATF', 'BATF3', 'CEBPB', 'CEBPZ', 'CREM', 'E2F4']


## Significant DEGs (rest vs act per lineage)

The PI specified: use "significant DEGs (padj < 0.05) comparing rest and act
for a particular cell type" as the target gene component of each genelist.

This DE file should be generated from a scanpy `rank_genes_groups` analysis
run separately (e.g. in a lightweight notebook without scPRINT2 loaded, to
avoid memory conflicts). The format expected is:

```
state,names,pvals_adj,logfoldchanges
CD4_rest,KLF2,1.2e-45,-2.3
CD4_act,FOSB,3.1e-32,4.1
...
```

Where `state` indicates which group was upregulated (e.g. CD4_rest = gene
is higher in rest than act for CD4 cells). Both up and downregulated genes
should be included — we want all genes that *differ* between states, not
just those upregulated in one direction.

**If you don't have this file yet:** see the instructions at the bottom
of this notebook for how to generate it in a separate Colab session.

In [ ]:
# ── CELL 10: Load pre-computed DE markers (rest vs act, per lineage) ────────────
# Expected columns: state, names, pvals_adj, logfoldchanges
# state values: 'CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act'
#   (indicating which group the gene is upregulated in)
#
# For each scPRINT2 state, we use the DEGs from BOTH directions of the comparison
# (i.e., both CD4_rest-upregulated and CD4_act-upregulated genes are included
# when building the genelist for either CD4 state). The point is to capture
# ALL genes that vary between states — the direction of change is what defines
# state specificity, but the full set of variable genes forms the best genelist.

de_cd4 = pd.read_csv(DE_CD4_PATH)
de_cd8 = pd.read_csv(DE_CD8_PATH, on_bad_lines='skip')
print(de_cd8.shape)
print(de_cd8.head())

# Filter to significant DEGs only (padj < 0.05)
de_cd4_sig = de_cd4[de_cd4['pvals_adj'] < 0.05]
de_cd8_sig = de_cd8[de_cd8['pvals_adj'] < 0.05]

# All significant CD4 DEGs (both rest-up and act-up), restricted to genes in dataset
de_cd4_genes = set(de_cd4_sig['names'].tolist()) & set(adata.var_names)
de_cd8_genes = set(de_cd8_sig['names'].tolist()) & set(adata.var_names)

print(f'Significant CD4 DEGs (in dataset): {len(de_cd4_genes)}')
print(f'Significant CD8 DEGs (in dataset): {len(de_cd8_genes)}')

# Quick sanity: do we see known markers?
known_act = {'FOSB', 'BATF', 'IRF4', 'JUN', 'JUNB', 'MYC', 'IL2RA'}
known_rest = {'KLF2', 'TCF7', 'LEF1', 'KLF3', 'FOXO1', 'CCR7', 'SELL'}
print(f'\nKnown activation markers in CD4 DEGs: {known_act & de_cd4_genes}')
print(f'Known resting markers in CD4 DEGs:    {known_rest & de_cd4_genes}')

(10026, 6)
     group     names     scores  logfoldchanges  pvals  pvals_adj
0  CD8_act  MIR155HG  47.467320        4.174674    0.0        0.0
1  CD8_act      GZMB  45.454536        2.802319    0.0        0.0
2  CD8_act      ENO1  44.791798        1.394242    0.0        0.0
3  CD8_act    NFKBIA  42.213240        1.956976    0.0        0.0
4  CD8_act  EEF1A1P5  41.548557        3.489017    0.0        0.0
Significant CD4 DEGs (in dataset): 4807
Significant CD8 DEGs (in dataset): 3782

Known activation markers in CD4 DEGs: {'JUNB', 'IL2RA', 'BATF', 'MYC', 'IRF4'}
Known resting markers in CD4 DEGs:    {'CCR7', 'TCF7', 'KLF3', 'SELL', 'LEF1', 'KLF2'}


In [ ]:
# ── CELL 11: Build state-specific genelists ───────────────────────────────────
# Per PI instructions:
#   CD4_rest genelist = pySCENIC resting TFs for CD4 (deltaRSS < 0)
#                     ∪ significant DEGs for CD4 rest vs act
#   CD4_act  genelist = pySCENIC activation TFs for CD4 (deltaRSS > 0)
#                     ∪ significant DEGs for CD4 rest vs act
#   CD8_rest genelist = pySCENIC resting TFs for CD8 (deltaRSS < 0)
#                     ∪ significant DEGs for CD8 rest vs act
#   CD8_act  genelist = pySCENIC activation TFs for CD8 (deltaRSS > 0)
#                     ∪ significant DEGs for CD8 rest vs act
#
# IMPORTANT: We use the *same DEG set* for both rest and act within a lineage.
# This is intentional — the DEGs define the variable gene space for that lineage's
# comparison. The TF component is what differs between the two states.

genelist_symbols_per_state = {
    'CD4_rest': sorted(pyscenic_tfs['CD4_rest'] | de_cd4_genes),
    'CD4_act':  sorted(pyscenic_tfs['CD4_act']  | de_cd4_genes),
    'CD8_rest': sorted(pyscenic_tfs['CD8_rest'] | de_cd8_genes),
    'CD8_act':  sorted(pyscenic_tfs['CD8_act']  | de_cd8_genes),
}

print('=== State-specific genelist sizes ===')
for state, gl in genelist_symbols_per_state.items():
    n_tfs    = len(all_tf_symbols & set(gl))
    n_nontf  = len(gl) - n_tfs
    print(f'{state:<12}: {len(gl):4d} genes total  ({n_tfs} TFs + {n_nontf} non-TF targets)')

# Sanity: check known resting/activation TFs landed in the right state lists
print('\n=== Sanity check: known TFs in correct state genelists ===')
print(f'KLF2 in CD4_rest genelist: {"KLF2" in set(genelist_symbols_per_state["CD4_rest"])}  '
      f'(expected True  — top resting TF)')
print(f'KLF2 in CD4_act  genelist: {"KLF2" in set(genelist_symbols_per_state["CD4_act"])}  '
      f'(expected False — not an activation TF)')
print(f'FOSB in CD4_act  genelist: {"FOSB" in set(genelist_symbols_per_state["CD4_act"])}  '
      f'(expected True  — top activation TF)')
print(f'FOSB in CD4_rest genelist: {"FOSB" in set(genelist_symbols_per_state["CD4_rest"])}  '
      f'(expected False — not a resting TF)')

=== State-specific genelist sizes ===
CD4_rest    : 4904 genes total  (338 TFs + 4566 non-TF targets)
CD4_act     : 4835 genes total  (269 TFs + 4566 non-TF targets)
CD8_rest    : 3917 genes total  (311 TFs + 3606 non-TF targets)
CD8_act     : 3788 genes total  (182 TFs + 3606 non-TF targets)

=== Sanity check: known TFs in correct state genelists ===
KLF2 in CD4_rest genelist: True  (expected True  — top resting TF)
KLF2 in CD4_act  genelist: True  (expected False — not an activation TF)
FOSB in CD4_act  genelist: True  (expected True  — top activation TF)
FOSB in CD4_rest genelist: False  (expected False — not a resting TF)


In [ ]:
# ── CELL 12: Populate bionty + Symbol → Ensembl conversion ────────────────────
# scPRINT2 uses Ensembl gene IDs internally. We convert all gene symbols to
# Ensembl IDs using the bionty gene table for human.
# populate_my_ontology() fills the local sqlite database (~1–5 min first call).

populate_my_ontology()

gene_df = bt.Gene.filter(organism__ontology_id='NCBITaxon:9606').to_dataframe(limit=None)
sym2ens = (
    gene_df.dropna(subset=['symbol', 'ensembl_gene_id'])
           .set_index('symbol')['ensembl_gene_id']
           .to_dict()
)
ens2sym = {v: k for k, v in sym2ens.items()}

# Convert the full adata.var_names (gene symbols) to Ensembl for alignment
new_var = [sym2ens.get(g) for g in adata.var_names]

# Convert each state's genelist to Ensembl, dropping unmappable symbols
genelist_ensembl_per_state = {}
for state, gl_sym in genelist_symbols_per_state.items():
    gl_ens = [sym2ens[s] for s in gl_sym if s in sym2ens]
    n_dropped = len(gl_sym) - len(gl_ens)
    genelist_ensembl_per_state[state] = gl_ens
    print(f'{state:<12}: {len(gl_sym)} symbols → {len(gl_ens)} Ensembl IDs '
          f'(dropped {n_dropped} unmappable)')

print('\nConversion complete ✓')

... synchronizing df_all__cl__2025-12-17__CellType.parquet: 100.0%
... synchronizing df_vertebrates__ensembl__release-114__Organism.parquet: 100.0%
! ambiguous validation in Bionty for 1 record: 'sheep'
! ambiguous validation in Bionty for 1 record: 'sheep'
! ontology ID BFO:0000020 not found in DataFrame
... synchronizing df_human__hancestro__2025-10-14__Ethnicity.parquet: 100.0%
... synchronizing df_all__efo__3.85.0__ExperimentalFactor.parquet: 100.0%
→ starting creation of 18326 ExperimentalFactor records in batches of 10000
→ starting creation of 16430 ExperimentalFactor_parents records in batches of 10000
! you are trying to create a record with name='unknown' but a record with similar name exists: 'fever of unknown origin'. Did you mean to load it?
... synchronizing df_all__uberon__2025-12-04__Tissue.parquet: 100.0%
→ starting creation of 15770 Tissue records in batches of 10000
→ starting creation of 42140 Tissue_parents records in batches of 10000
... synchronizing df_human__hs

In [ ]:
# ── CELL 13: Download scPRINT2 model checkpoint (if not cached) ───────────────
# The small-v2.ckpt is the recommended checkpoint for GRN inference.
# ~500 MB download from HuggingFace. Cached in /content after first download.

if not os.path.exists(CKPT_PATH):
    print('Downloading small-v2.ckpt from HuggingFace...')
    hf_hub_download(repo_id='jkobject/scPRINT', filename='small-v2.ckpt',
                    local_dir='/content')
    print('Downloaded ✓')
else:
    print(f'Model checkpoint cached ✓  ({os.path.getsize(CKPT_PATH)/1024**2:.0f} MB)')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


small-v2.ckpt:   0%|          | 0.00/890M [00:00<?, ?B/s]

Downloaded ✓


In [ ]:
# ── CELL 14: Load scPRINT2 model ─────────────────────────────────────────────
from scprint2 import scPRINT2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = scPRINT2.load_from_checkpoint(CKPT_PATH, precpt_gene_emb=None, gene_pos_file=None)
if device == 'cpu':
    model = model.to(torch.float32)  # CPU requires float32 (no bfloat16 support)
model = model.to(device)
model.organisms = ['NCBITaxon:9606']
print(f'Model loaded on {device} ✓')
print(f'Model gene universe size: {len(model.genes)}')
print(f'Model layers: {model.nlayers}')

No module named 'adasplash'
FlashAttention is not installed, not using it..


/usr/local/lib/python3.12/dist-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /usr/local/lib/python3.12/dist-packages
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


FYI: scPRINT2 is not attached to a `Trainer`.
Model loaded on cuda ✓
Model gene universe size: 20004
Model layers: 8


## Preparing Per-State AnnData Objects

scPRINT2's DataCollator requires:
1. Cells expressed over the model's ~19k–85k-gene Ensembl universe (zero-padded for missing genes)
2. A precomputed neighbor graph (k-NN connectivity matrix in `adata.obsp`)

We process each of the four states independently to:
- Subsample to `CELLS_PER_STATE` cells (memory-efficient)
- Normalize → log1p → PCA → neighbors on just those cells
- Align to the model gene universe
- Expand to the full Collator universe with zero-padding

This is the same strategy as `scprint_grn_clean.ipynb`, preserved here.

In [ ]:
# ── CELL 15: Build per-state AnnData objects with real neighbor graphs ────────
full_gene_list = load_genes(organisms=['NCBITaxon:9606']).index.tolist()
model_genes    = set(model.genes)

adata_full_per_state = {}

for state in STATES:
    print(f'\n── Preparing {state} ──')

    # 1. Subset to this cell state
    sub = adata[adata.obs['group4'] == state].copy()
    print(f'  Total cells in state: {sub.n_obs}')

    # 2. Subsample to CELLS_PER_STATE for memory efficiency
    #    (we don't need all 10–15k cells; 2000 gives stable neighbor graphs)
    if sub.n_obs > CELLS_PER_STATE:
        sc.pp.subsample(sub, n_obs=CELLS_PER_STATE, random_state=42)
    print(f'  After subsample: {sub.n_obs} cells')

    # 3. Normalize + log1p + PCA + neighbors
    #    Normalize to 10,000 counts/cell (library-size normalization)
    #    Log1p to stabilize variance
    #    PCA for dimensionality reduction before neighbor graph
    #    Neighbors = k-NN graph required by scPRINT2's DataCollator
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)
    sc.pp.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, use_rep='X_pca')
    print(f'  Normalize → log1p → PCA → neighbors ✓')

    # 4. Align to model gene universe (Ensembl IDs)
    #    The model only knows its training gene universe; genes not in it must be dropped.
    sub_new_var = [sym2ens.get(g) for g in sub.var_names]
    mask        = [x is not None and x in model_genes for x in sub_new_var]
    sub_aligned = sub[:, mask].copy()
    sub_aligned.var_names = [sub_new_var[i] for i, m in enumerate(mask) if m]
    print(f'  Genes overlapping model universe: {sub_aligned.n_vars}')

    # 5. Expand to full Collator gene universe, zero-padding missing genes
    #    The DataCollator internally indexes into a fixed gene list ~85k long;
    #    we must provide a matrix of exactly that width.
    idx_map  = {g: i for i, g in enumerate(sub_aligned.var_names)}
    col_map  = [idx_map.get(g) for g in full_gene_list]
    in_full  = [j for j, v in enumerate(col_map) if v is not None]
    src_cols = [col_map[j] for j in in_full]

    X_orig = sp.csr_matrix(sub_aligned.X)
    X_exp  = sp.lil_matrix((sub_aligned.n_obs, len(full_gene_list)), dtype=np.float32)
    X_exp[:, in_full] = X_orig[:, src_cols]

    sub_full = ad.AnnData(X=X_exp.tocsr(), obs=sub_aligned.obs.copy())
    sub_full.var_names = full_gene_list

    # Carry over the real neighbor graph from the subsampled subset
    sub_full.obsp['connectivities'] = sub.obsp['connectivities']
    sub_full.obsp['distances']      = sub.obsp['distances']
    sub_full.uns['neighbors']       = sub.uns['neighbors']

    # Annotate each gene with its symbol and TF status
    sub_full.var['symbol'] = [ens2sym.get(g, g) for g in sub_full.var_names]
    sub_full.var['isTF']   = sub_full.var['symbol'].isin(grnutils.TF)

    adata_full_per_state[state] = sub_full
    print(f'  adata_full shape: {sub_full.shape} | TFs in var: {sub_full.var.isTF.sum()}')

    # Free intermediate objects before next state
    del sub, sub_aligned
    gc.collect()

print('\nAll states prepared ✓')


── Preparing CD4_rest ──
  Total cells in state: 16808
  After subsample: 2000 cells
  Normalize → log1p → PCA → neighbors ✓
  Genes overlapping model universe: 18027
  adata_full shape: (2000, 85720) | TFs in var: 1620

── Preparing CD4_act ──
  Total cells in state: 18144
  After subsample: 2000 cells
  Normalize → log1p → PCA → neighbors ✓
  Genes overlapping model universe: 18027
  adata_full shape: (2000, 85720) | TFs in var: 1620

── Preparing CD8_rest ──
  Total cells in state: 6669
  After subsample: 2000 cells
  Normalize → log1p → PCA → neighbors ✓
  Genes overlapping model universe: 18027
  adata_full shape: (2000, 85720) | TFs in var: 1620

── Preparing CD8_act ──
  Total cells in state: 6105
  After subsample: 2000 cells
  Normalize → log1p → PCA → neighbors ✓
  Genes overlapping model universe: 18027
  adata_full shape: (2000, 85720) | TFs in var: 1620

All states prepared ✓


## GNInfer Setup — Bug Patch

scPRINT2 has a known bug in `GNInfer.__init__`: `self.genelist` is accessed inside
`__init__` before it is assigned, causing an `AttributeError`. The fix is to:
1. Reload the module to get a clean `__init__` (avoids stale monkey-patches)
2. Prepend an assignment of `self.genelist` before calling the original `__init__`

**Do NOT re-run this cell** — applying the patch twice causes infinite recursion.
If you need to re-run it, first do `importlib.reload(scprint2.tasks.grn)` in a fresh cell
to reset `GNInfer.__init__` to the original before patching again.

In [ ]:
# ── CELL 16: Patch GNInfer and instantiate (run ONCE only) ───────────────────
# ⚠️  Do NOT re-run this cell. If you need to re-run, reload the module first:
#     import importlib, scprint2.tasks.grn; importlib.reload(scprint2.tasks.grn)
# Then re-run this cell.

import importlib
import scprint2.tasks.grn
importlib.reload(scprint2.tasks.grn)   # ensures we get a clean __init__
from scprint2.tasks.grn import GNInfer

_clean_init = GNInfer.__init__

def _patched_init(self, *args, **kwargs):
    # Set self.genelist BEFORE calling the original __init__, which tries to read it.
    self.genelist = kwargs.get('genelist', [])
    _clean_init(self, *args, **kwargs)

GNInfer.__init__ = _patched_init
print('GNInfer patched ✓')

# We instantiate ONE GNInfer object per state because each state has a different genelist.
# The genelist is set at construction time and used internally to select which gene
# entries of the attention matrix to return.
#
# Parameters:
#   how='some'         → use the provided genelist as the gene universe for this run
#   preprocess='softmax' → softmax-normalize the raw attention weights (standard)
#   head_agg='mean'    → average across all attention heads (reduces noise)
#   filtration='none'  → no post-hoc edge filtering (we do our own downstream)
#   max_cells=MAX_CELLS → subsample to this many cells for GPU inference
#   drop_unexpressed=True → within the cell subset, drop genes with zero expression
#   layer=all_layers   → aggregate attention from all transformer layers

grn_inferers = {}
for state in STATES:
    grn_inferers[state] = GNInfer(
        how='some',
        genelist=genelist_ensembl_per_state[state],
        preprocess='softmax',
        head_agg='mean',
        filtration='none',
        max_cells=MAX_CELLS,
        drop_unexpressed=True,
        batch_size=BATCH_SIZE,
        cell_type_col='group4',
        layer=list(range(model.nlayers)),
    )
    print(f'{state:<12}: GNInfer ready | genelist: {len(genelist_ensembl_per_state[state])} Ensembl genes')

GNInfer patched ✓
CD4_rest    : GNInfer ready | genelist: 4504 Ensembl genes
CD4_act     : GNInfer ready | genelist: 4434 Ensembl genes
CD8_rest    : GNInfer ready | genelist: 3644 Ensembl genes
CD8_act     : GNInfer ready | genelist: 3516 Ensembl genes


In [ ]:
# ── CELL 17: Run GRN inference for all 4 states ───────────────────────────────
# Each state uses its own GNInfer (state-specific genelist) and its own
# pre-built adata_full with real neighbors.
#
# Outputs are written directly to Drive immediately after each state completes,
# so a runtime disconnect only loses states not yet processed.
#
# Output format: AnnData (.h5ad) where
#   grn.varp['GRN'] = [n_genes × n_genes] attention weight matrix
#   grn.var_names   = Ensembl IDs of genes in the GRN
#   grn.var['symbol'] = gene symbol for each row/column
#   grn.var['isTF']   = whether this gene is a TF (Lambert et al. 2018)

grns = {}

for state in STATES:
    print(f'\n=== {state} ===')
    sub_full = adata_full_per_state[state]
    print(f'  Input cells: {sub_full.n_obs} | Input genes: {sub_full.n_vars}')

    if sub_full.n_obs < 20:
        print('  SKIPPING — fewer than 20 cells in this state')
        continue

    # Run GRN inference
    grn = grn_inferers[state](model, sub_full, cell_type=state)

    # Annotate output with gene symbols and TF status
    grn.var['symbol'] = [ens2sym.get(g, g) for g in grn.var_names]
    grn.var['isTF']   = grn.var['symbol'].isin(grnutils.TF)
    n_tfs = int(grn.var.isTF.sum())

    # Save to Drive immediately — do not wait until the loop finishes
    out_path = os.path.join(GRN_OUT_DIR, f'{state}_grn_pi_revised.h5ad')
    grn.write_h5ad(out_path)
    grns[state] = grn

    print(f'  ✓ Saved → {out_path}')
    print(f'  GRN shape: {grn.varp["GRN"].shape} | TFs in output: {n_tfs}')

    # Free GPU memory before the next state
    del sub_full
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nInference complete ✓  States processed: {list(grns.keys())}')


=== CD4_rest ===
  Input cells: 2000 | Input genes: 85720
Some valid genes are not in the genedf!!!
predict epoch start


100%|██████████| 19/19 [00:04<00:00,  4.69it/s]


avg link count: 18071001, sparsity: 1.0
  ✓ Saved → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3/CD4_rest_grn_pi_revised.h5ad
  GRN shape: (4251, 4251) | TFs in output: 333

=== CD4_act ===
  Input cells: 2000 | Input genes: 85720
Some valid genes are not in the genedf!!!
predict epoch start


100%|██████████| 19/19 [00:03<00:00,  5.83it/s]


avg link count: 17489124, sparsity: 1.0
  ✓ Saved → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3/CD4_act_grn_pi_revised.h5ad
  GRN shape: (4182, 4182) | TFs in output: 263

=== CD8_rest ===
  Input cells: 2000 | Input genes: 85720
Some valid genes are not in the genedf!!!
predict epoch start


100%|██████████| 19/19 [00:03<00:00,  6.04it/s]


avg link count: 12159169, sparsity: 1.0
  ✓ Saved → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3/CD8_rest_grn_pi_revised.h5ad
  GRN shape: (3487, 3487) | TFs in output: 301

=== CD8_act ===
  Input cells: 2000 | Input genes: 85720
Some valid genes are not in the genedf!!!
predict epoch start


100%|██████████| 19/19 [00:03<00:00,  6.02it/s]


avg link count: 11343424, sparsity: 1.0
  ✓ Saved → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3/CD8_act_grn_pi_revised.h5ad
  GRN shape: (3368, 3368) | TFs in output: 181

Inference complete ✓  States processed: ['CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act']


## Matrix Directionality — Verification

Before extracting edges, we verify which axis of the GRN matrix is the regulator
(TF) and which is the target. This is critical for biological interpretation.

**scPRINT2 attention convention:**
In a transformer, the attention weight A[i, j] = how much position i (query)
attends to position j (key). In the GRN context:
- Gene i (row) is the **target** — the gene whose expression is being predicted
- Gene j (column) is the **regulator** — the gene providing regulatory signal

So: `GRN[i, j]` = gene j regulates gene i → **columns are TFs, rows are targets**

We verify this by checking asymmetry (a purely symmetric matrix would suggest
the model is not capturing direction) and by looking at whether biologically
known TF→target relationships appear in the correct orientation.

In [ ]:
# ── CELL 18: Matrix directionality verification ───────────────────────────────
# Run this before edge extraction to confirm column=regulator, row=target.

print('=== DIRECTIONALITY CHECK ===')
print()
print('scPRINT2 attention convention:')
print('  GRN[i, j] = attention from gene i (query/target) to gene j (key/regulator)')
print('  → COLUMNS are regulators (TFs), ROWS are targets')
print()

for state, grn in grns.items():
    mat     = grn.varp['GRN']
    symbols = grn.var['symbol'].values
    is_tf   = grn.var['isTF'].values
    dense   = mat.toarray() if sp.issparse(mat) else np.array(mat)

    # 1. Symmetry check: a perfectly symmetric matrix has no directionality
    sym_diff = np.abs(dense - dense.T).mean()
    print(f'{state}: mean |A - A^T| = {sym_diff:.6f}  '
          f'(0 = fully symmetric / no direction, >0 = directed ✓)')

    # 2. Check if TF columns sum > TF rows sum
    #    If columns=regulators, TF COLUMNS should have higher total weight
    #    than TF ROWS (since TFs regulate many genes but are not uniquely targeted)
    tf_col_sum = dense[:, is_tf].sum()
    tf_row_sum = dense[is_tf, :].sum()
    print(f'  TF column sum: {tf_col_sum:.2f}  |  TF row sum: {tf_row_sum:.2f}')
    if tf_col_sum > tf_row_sum:
        print(f'  ✓ TF columns outweigh TF rows — confirms columns = regulators')
    else:
        print(f'  ⚠ TF rows outweigh TF columns — convention may be inverted; check GNInfer source')

    print()

print('CONCLUSION: if the checks above pass, use:')
print('  rows, cols = np.where(... & np.outer(~is_tf, is_tf))')
print('  edges["TF"]     = symbols[cols]   # column = regulator')
print('  edges["target"] = symbols[rows]   # row    = target')

=== DIRECTIONALITY CHECK ===

scPRINT2 attention convention:
  GRN[i, j] = attention from gene i (query/target) to gene j (key/regulator)
  → COLUMNS are regulators (TFs), ROWS are targets

CD4_rest: mean |A - A^T| = 0.000303  (0 = fully symmetric / no direction, >0 = directed ✓)
  TF column sum: 261.25  |  TF row sum: 333.00
  ⚠ TF rows outweigh TF columns — convention may be inverted; check GNInfer source

CD4_act: mean |A - A^T| = 0.000295  (0 = fully symmetric / no direction, >0 = directed ✓)
  TF column sum: 209.28  |  TF row sum: 263.00
  ⚠ TF rows outweigh TF columns — convention may be inverted; check GNInfer source

CD8_rest: mean |A - A^T| = 0.000367  (0 = fully symmetric / no direction, >0 = directed ✓)
  TF column sum: 277.32  |  TF row sum: 301.00
  ⚠ TF rows outweigh TF columns — convention may be inverted; check GNInfer source

CD8_act: mean |A - A^T| = 0.000363  (0 = fully symmetric / no direction, >0 = directed ✓)
  TF column sum: 154.80  |  TF row sum: 181.00
  ⚠ TF r

In [ ]:
# ── CELL 19: Extract TF→target edge tables ────────────────────────────────────
# Flatten the [n_genes × n_genes] attention weight matrix into a long-format
# edge list: (TF, target, weight, state).
#
# Convention (verified in Cell 18):
#   GRN[i, j] = weight of gene j regulating gene i
#   → ROWS are targets, COLUMNS are regulators (TFs)
#
# So we select cells where:
#   - The COLUMN gene (j) is a TF         → is_tf[col] = True
#   - The ROW gene (i) is NOT a TF        → is_tf[row] = False  (TF → non-TF target)
#   - Weight exceeds WEIGHT_THRESHOLD
#   - Not a self-loop
#
# If you also want TF→TF edges (i.e., one TF regulating another TF),
# remove the ~is_tf mask on the row side.

all_edges = {}

for state, grn in grns.items():
    mat     = grn.varp['GRN']
    symbols = grn.var['symbol'].values
    is_tf   = grn.var['isTF'].values
    dense   = mat.toarray() if sp.issparse(mat) else np.array(mat)

    # Select entries: weight > threshold AND column is TF AND row is not TF
    # np.outer(~is_tf, is_tf) creates a boolean mask of shape [n_genes × n_genes]
    # where True means: row is non-TF target AND column is TF regulator
    mask_edges = (dense > WEIGHT_THRESHOLD) & np.outer(~is_tf, is_tf)
    rows, cols = np.where(mask_edges)

    edges = pd.DataFrame({
        'TF':     symbols[cols],   # column = regulator (TF)
        'target': symbols[rows],   # row    = target gene
        'weight': dense[rows, cols],
        'state':  state,
    })

    # Remove self-loops (should not exist given the mask, but belt-and-suspenders)
    edges = edges[edges['TF'] != edges['target']]

    all_edges[state] = edges

    out_csv = os.path.join(GRN_OUT_DIR, f'{state}_scprint_pi_revised_edges.csv')
    edges.to_csv(out_csv, index=False)
    print(f'{state:<12}: {len(edges):6,d} TF→target edges → {out_csv}')

# Combined table for benchmarking
combined = pd.concat(all_edges.values(), ignore_index=True)
combined.to_csv(os.path.join(GRN_OUT_DIR, 'all_states_scprint_pi_revised_edges.csv'), index=False)
print(f'\nCombined: {len(combined):,} total edges → all_states_scprint_pi_revised_edges.csv')

CD4_rest    :    185 TF→target edges → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3/CD4_rest_scprint_pi_revised_edges.csv
CD4_act     :    105 TF→target edges → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3/CD4_act_scprint_pi_revised_edges.csv
CD8_rest    :    935 TF→target edges → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3/CD8_rest_scprint_pi_revised_edges.csv
CD8_act     :    522 TF→target edges → /content/drive/MyDrive/benchmarking_paper/datasets/scPRINT/grn_outputs_pi_revisedv3/CD8_act_scprint_pi_revised_edges.csv

Combined: 1,747 total edges → all_states_scprint_pi_revised_edges.csv


In [ ]:
# ── CELL 20: QC summary ───────────────────────────────────────────────────────
print('=== TF Coverage per State ===')
print(f'{"State":<12} {"Genes in GRN":<15} {"TFs":<8} {"TF%"}')
print('-' * 42)
for state, grn in grns.items():
    n_g = grn.n_vars
    n_t = int(grn.var.isTF.sum())
    print(f'{state:<12} {n_g:<15} {n_t:<8} {100*n_t/n_g:.1f}%')

print('\n=== Top 20 TFs by total outgoing regulatory weight per state ===')
for state, edges in all_edges.items():
    top = (edges.groupby('TF')['weight'].sum()
                .sort_values(ascending=False)
                .head(20))
    print(f'\n{state}:')
    print(top.to_string())

# Cross-check: do known biology TFs appear with correct sign?
print('\n=== Biological sanity check ===')
for state in STATES:
    edges = all_edges.get(state, pd.DataFrame())
    if edges.empty: continue
    tf_score = edges.groupby('TF')['weight'].sum().sort_values(ascending=False)
    print(f'\n{state}:')
    for known_tf in ['FOSB', 'KLF2', 'IRF4', 'BATF', 'TCF7', 'EGR2']:
        score = tf_score.get(known_tf, None)
        rank  = (tf_score.index.tolist().index(known_tf) + 1) if known_tf in tf_score.index else 'absent'
        print(f'  {known_tf:<10}: score={score:.4f if score else "N/A"}, rank={rank}')

=== TF Coverage per State ===
State        Genes in GRN    TFs      TF%
------------------------------------------
CD4_rest     4251            333      7.8%
CD4_act      4182            263      6.3%
CD8_rest     3487            301      8.6%
CD8_act      3368            181      5.4%

=== Top 20 TFs by total outgoing regulatory weight per state ===

CD4_rest:
TF
GPBP1     1.052344
PREB      0.413277
MEF2C     0.376030
SREBF1    0.342349
THAP1     0.058446
NR3C1     0.011180

CD4_act:
TF
GPBP1     1.104491
ZNF266    0.113076
NR3C1     0.056244
THAP1     0.043594

CD8_rest:
TF
GPBP1      6.864486
ZNF266     3.707529
PREB       2.088476
ZNF780A    0.153051
SREBF1     0.065068
THAP1      0.020914
GFI1B      0.011345

CD8_act:
TF
FOSB     4.273581
DEAF1    1.121371
GPBP1    1.026287

=== Biological sanity check ===

CD4_rest:


TypeError: unsupported format string passed to NoneType.__format__

## Appendix: How to Generate DE Markers (Separate Notebook)

Because `scanpy` + `scPRINT2` together can run into memory conflicts, generate
DE markers in a **lightweight separate Colab session** (no scPRINT2 needed):

```python
import scanpy as sc, pandas as pd

# Load the raw data from Dropbox (or the HVG-filtered Drive file — Wilcoxon works on both)
adata = sc.read_h5ad('/content/tcell_cache/tcell_raw.h5ad')
adata.obs['group4'] = adata.obs['cd_status'].astype(str) + '_' + adata.obs['stimulation'].astype(str)

# Normalize for DE (Wilcoxon is rank-based but normalization avoids library-size bias)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ── CD4 rest vs act ────────────────────────────────────────────────────────
cd4 = adata[adata.obs['group4'].isin(['CD4_rest', 'CD4_act'])].copy()
sc.tl.rank_genes_groups(cd4, groupby='group4', groups=['CD4_rest', 'CD4_act'],
                         method='wilcoxon', reference='rest',  # act compared to rest
                         key_added='de_cd4')
de_cd4 = sc.get.rank_genes_groups_df(cd4, group=None, key='de_cd4')
de_cd4.to_csv('/content/de_cd4_rest_vs_act.csv', index=False)
print('CD4 DE done:', de_cd4.shape)

# ── CD8 rest vs act ────────────────────────────────────────────────────────
cd8 = adata[adata.obs['group4'].isin(['CD8_rest', 'CD8_act'])].copy()
sc.tl.rank_genes_groups(cd8, groupby='group4', groups=['CD8_rest', 'CD8_act'],
                         method='wilcoxon', reference='rest',
                         key_added='de_cd8')
de_cd8 = sc.get.rank_genes_groups_df(cd8, group=None, key='de_cd8')
de_cd8.to_csv('/content/de_cd8_rest_vs_act.csv', index=False)
print('CD8 DE done:', de_cd8.shape)
```

Then **download** `de_cd4_rest_vs_act.csv` and `de_cd8_rest_vs_act.csv` from that session
and **upload** them to the scPRINT2 session before running Cell 10.

Expected output columns: `group, names, scores, logfoldchanges, pvals, pvals_adj, pts, pts_rest`
Cell 10 uses `names` and `pvals_adj` columns.

---
## Appendix: If You Already Have `all_states_scprint_pi_revised_edges.csv`

You can skip to benchmarking without re-running inference:

```python
import pandas as pd
df = pd.read_csv(
    '/content/drive/MyDrive/benchmarking_paper/datasets/'
    'scPRINT/grn_outputs_pi_revised/all_states_scprint_pi_revised_edges.csv'
)
print(df.head())
print(df.groupby('state').size())
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# T-CELL GRN BENCHMARKING ANALYSIS
# Loads all three networks and runs structural comparison
# ═══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/benchmarking_paper/datasets'
STATES = ['CD4_rest', 'CD4_act', 'CD8_rest', 'CD8_act']

# ── 1. LOAD ALL THREE NETWORKS ───────────────────────────────────────────────

# pySCENIC-net
scenic = {}
for state in STATES:
    scenic[state] = pd.read_csv(f'{BASE}/scenic_net/scenic_net_{state}.csv')

# scPRINT2-net (PI revised)
scprint = {}
for state in STATES:
    scprint[state] = pd.read_csv(
        f'{BASE}/scPRINT/grn_outputs_pi_revised/{state}_scprint_pi_revised_edges.csv'
    )

# perturb-net (split by cell state if you have it, otherwise load combined)
perturb = pd.read_csv(f'{BASE}/perturb_net/perturb_net_tfs_only.csv')
# Split perturb-net by state if it has a state/stimulation column
# Adjust column name to match your actual file
if 'cell_state' in perturb.columns:
    perturb_by_state = {s: perturb[perturb['cell_state'] == s] for s in STATES}
else:
    # If no state split yet, use the whole thing for all states
    perturb_by_state = {s: perturb for s in STATES}

print('=== FILES LOADED ===')
for state in STATES:
    print(f'\n{state}:')
    print(f'  pySCENIC : {len(scenic[state]):6,} edges | {scenic[state]["TF"].nunique()} TFs')
    print(f'  scPRINT2 : {len(scprint[state]):6,} edges | {scprint[state]["TF"].nunique()} TFs')
    print(f'  perturb  : {len(perturb_by_state[state]):6,} edges | {perturb_by_state[state]["TF"].nunique()} TFs')


# ── 2. SHARED TFs ACROSS ALL THREE NETWORKS ──────────────────────────────────
print('\n=== TF OVERLAP ACROSS NETWORKS (per state) ===')
for state in STATES:
    tfs_scenic  = set(scenic[state]['TF'])
    tfs_scprint = set(scprint[state]['TF'])
    tfs_perturb = set(perturb_by_state[state]['TF'])

    all_three = tfs_scenic & tfs_scprint & tfs_perturb
    scenic_scprint = tfs_scenic & tfs_scprint
    scenic_perturb = tfs_scenic & tfs_perturb
    scprint_perturb = tfs_scprint & tfs_perturb

    print(f'\n{state}:')
    print(f'  pySCENIC ∩ scPRINT2 ∩ perturb  : {sorted(all_three)}')
    print(f'  pySCENIC ∩ scPRINT2             : {len(scenic_scprint)} TFs')
    print(f'  pySCENIC ∩ perturb              : {len(scenic_perturb)} TFs')
    print(f'  scPRINT2 ∩ perturb              : {len(scprint_perturb)} TFs')


# ── 3. EDGE OVERLAP PER TF (the core benchmarking metric) ───────────────────
# For each TF, what fraction of its edges are shared between networks?
# This is the key signal for identifying high-confidence TF→target pairs.

def get_edge_set(df, tf_col='TF', target_col='target'):
    """Return a set of (TF, target) tuples from an edge dataframe."""
    return set(zip(df[tf_col], df[target_col]))

print('\n=== EDGE OVERLAP PER TF ===')
overlap_records = []

for state in STATES:
    edges_scenic  = get_edge_set(scenic[state])
    edges_scprint = get_edge_set(scprint[state])
    edges_perturb = get_edge_set(perturb_by_state[state])

    # All TFs that appear in at least one network for this state
    all_tfs = (set(scenic[state]['TF']) |
               set(scprint[state]['TF']) |
               set(perturb_by_state[state]['TF']))

    for tf in all_tfs:
        # Edges for this TF in each network
        sc_edges = {(t, g) for t, g in edges_scenic  if t == tf}
        sp_edges = {(t, g) for t, g in edges_scprint if t == tf}
        pb_edges = {(t, g) for t, g in edges_perturb if t == tf}

        overlap_records.append({
            'state':             state,
            'TF':                tf,
            'n_scenic':          len(sc_edges),
            'n_scprint':         len(sp_edges),
            'n_perturb':         len(pb_edges),
            'scenic_scprint':    len(sc_edges & sp_edges),
            'scenic_perturb':    len(sc_edges & pb_edges),
            'scprint_perturb':   len(sp_edges & pb_edges),
            'all_three':         len(sc_edges & sp_edges & pb_edges),
            'in_scenic':         len(sc_edges) > 0,
            'in_scprint':        len(sp_edges) > 0,
            'in_perturb':        len(pb_edges) > 0,
        })

overlap_df = pd.DataFrame(overlap_records)

# TFs present in all three networks — highest confidence
high_conf = overlap_df[
    overlap_df['in_scenic'] & overlap_df['in_scprint'] & overlap_df['in_perturb']
]
print(f'\nTFs present in ALL THREE networks:')
print(high_conf.groupby('state')['TF'].apply(sorted).to_string())


# ── 4. HUB TF IDENTIFICATION ─────────────────────────────────────────────────
# Hub TFs = high out-degree (many target genes) across networks
# These are likely master regulators of the T cell state

print('\n=== HUB TFs (top 10 by total edges across networks) ===')
for state in STATES:
    combined = pd.concat([
        scenic[state][['TF', 'target']],
        scprint[state][['TF', 'target']],
        perturb_by_state[state][['TF', 'target']],
    ])
    hub = combined.groupby('TF')['target'].nunique().sort_values(ascending=False).head(10)
    print(f'\n{state}:')
    print(hub.to_string())


# ── 5. REWIRED EDGES: REST → ACTIVATION ─────────────────────────────────────
# Which TF→target edges appear in activation but not rest (gained)?
# Which appear in rest but not activation (lost)?
# Do this for scPRINT2 (most sensitive to state-specific rewiring)

print('\n=== REWIRED EDGES (scPRINT2) ===')
for lineage in ['CD4', 'CD8']:
    rest_edges = get_edge_set(scprint[f'{lineage}_rest'])
    act_edges  = get_edge_set(scprint[f'{lineage}_act'])

    gained = act_edges - rest_edges
    lost   = rest_edges - act_edges

    print(f'\n{lineage}:')
    print(f'  Edges GAINED on activation: {len(gained)}')
    print(f'  Edges LOST on activation:   {len(lost)}')

    # Which TFs drive the most gained/lost edges?
    gained_df = pd.DataFrame(list(gained), columns=['TF', 'target'])
    lost_df   = pd.DataFrame(list(lost),   columns=['TF', 'target'])

    print(f'  Top TFs gaining edges:')
    print(gained_df['TF'].value_counts().head(5).to_string())
    print(f'  Top TFs losing edges:')
    print(lost_df['TF'].value_counts().head(5).to_string())


# ── 6. REGULATORY ACTIVITY DELTA (scPRINT2 weight difference) ───────────────
# For TFs present in both rest and act, how much does their total
# outgoing weight change? Positive = more active in act, negative = more in rest.

print('\n=== REGULATORY ACTIVITY DELTA (scPRINT2, act - rest weight) ===')
for lineage in ['CD4', 'CD8']:
    rest_scores = (scprint[f'{lineage}_rest']
                   .groupby('TF')['weight'].sum()
                   .rename('rest'))
    act_scores  = (scprint[f'{lineage}_act']
                   .groupby('TF')['weight'].sum()
                   .rename('act'))

    delta = pd.concat([rest_scores, act_scores], axis=1).dropna()
    delta['delta'] = delta['act'] - delta['rest']
    delta = delta.sort_values('delta', ascending=False)

    print(f'\n{lineage} — Top activation TFs (delta > 0):')
    print(delta[delta['delta'] > 0].head(10)[['rest', 'act', 'delta']].to_string())
    print(f'\n{lineage} — Top resting TFs (delta < 0):')
    print(delta[delta['delta'] < 0].tail(10)[['rest', 'act', 'delta']].to_string())


# ── 7. SUMMARY TABLE — HIGH CONFIDENCE CORE NETWORK CANDIDATES ──────────────
# TFs that are: (a) in all 3 networks, (b) hub TFs, (c) show state-specific delta
# These are your best T-net candidates

print('\n=== HIGH CONFIDENCE T-NET CANDIDATES ===')
for state in STATES:
    state_overlap = overlap_df[
        (overlap_df['state'] == state) &
        overlap_df['in_scenic'] &
        overlap_df['in_scprint'] &
        overlap_df['in_perturb']
    ].sort_values('all_three', ascending=False)

    print(f'\n{state} ({len(state_overlap)} TFs in all 3 networks):')
    print(state_overlap[['TF', 'n_scenic', 'n_scprint', 'n_perturb', 'all_three']].to_string(index=False))


# ── 8. QUICK PLOTS ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('T-Cell GRN Network Comparison', fontsize=14, fontweight='bold')

for ax, state in zip(axes.flat, STATES):
    state_df = overlap_df[overlap_df['state'] == state]

    # Venn-style bar: how many TFs in 1, 2, or 3 networks
    counts = {
        'scenic only':          len(state_df[state_df['in_scenic'] & ~state_df['in_scprint'] & ~state_df['in_perturb']]),
        'scprint only':         len(state_df[~state_df['in_scenic'] & state_df['in_scprint'] & ~state_df['in_perturb']]),
        'perturb only':         len(state_df[~state_df['in_scenic'] & ~state_df['in_scprint'] & state_df['in_perturb']]),
        'scenic+scprint':       len(state_df[state_df['in_scenic'] & state_df['in_scprint'] & ~state_df['in_perturb']]),
        'scenic+perturb':       len(state_df[state_df['in_scenic'] & ~state_df['in_scprint'] & state_df['in_perturb']]),
        'scprint+perturb':      len(state_df[~state_df['in_scenic'] & state_df['in_scprint'] & state_df['in_perturb']]),
        'all three':            len(state_df[state_df['in_scenic'] & state_df['in_scprint'] & state_df['in_perturb']]),
    }
    colors = ['#d9d9d9','#d9d9d9','#d9d9d9','#9ecae1','#9ecae1','#9ecae1','#2166ac']
    ax.barh(list(counts.keys()), list(counts.values()), color=colors)
    ax.set_title(state)
    ax.set_xlabel('Number of TFs')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/benchmarking_paper/figures/tf_overlap_summary.png', dpi=150)
plt.show()
print('Plot saved ✓')

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/benchmarking_paper/datasets'

# pySCENIC
scenic = pd.read_csv(f'{BASE}/scenic_net/scenic_net_CD4_rest.csv')
print('=== pySCENIC ===')
print(scenic.shape)
print(scenic.head(3))

# scPRINT2
scprint = pd.read_csv(f'{BASE}/scPRINT/grn_outputs_pi_revised/CD4_rest_scprint_pi_revised_edges.csv')
print('\n=== scPRINT2 ===')
print(scprint.shape)
print(scprint.head(3))

# perturb-net
perturb = pd.read_csv(f'{BASE}/perturb_net/perturb_net_tfs_only.csv')
print('\n=== perturb-net ===')
print(perturb.shape)
print(perturb.head(3))

In [ ]:
print(scenic['CD4_rest'].columns.tolist())
print(scprint['CD4_rest'].columns.tolist())
print(perturb.columns.tolist())